# Revision analyses R3 — Appendix A.3 tables and figures, revised Figure 6, consistency checks
Builds the supplementary tables and figures cited in the revised manuscript and response letter, plus a revised Figure 6, **from outputs already on Drive (no retraining; ~10–15 min)**. Also runs two consistency checks:

* **Check A** — Table 2's linear-interpolation row (two different calculations were mixed; one silently drops a stalagmite with no depth values).
* **Check B** — whether the tuned hyper-parameters are identical in the LOCO and within-cave runs.

Run R1 and R2 first. Outputs: `artifacts/revision_2026_11/supplement/`.


## 0 · Setup
Mounts Drive, defines paths, constants and the helpers used throughout. Run all cells in order.
Everything is written to `artifacts/revision_2026_11/` on Drive; nothing in the original run folders is modified.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import subprocess, sys, importlib, importlib.util
def _pip(*pkgs):
    for p in pkgs:
        mod = p.split("==")[0].split(">=")[0].replace("-", "_")
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)
_pip("catboost", "xgboost")

In [ ]:
import os, json, time, warnings, zlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

# ---- Paths -----------------------------------------------------------------------
_candidates = [Path(os.environ["SISAL_BASE"])] if "SISAL_BASE" in os.environ else []
_candidates += [Path("/content/drive/MyDrive/SISALv3_Speleothem"),
                Path("/content/drive/My Drive/SISALv3_Speleothem")]
base = next((p for p in _candidates if p.exists()), _candidates[0])
ARTIFACTS   = base / "artifacts"
LOCO_DIR    = ARTIFACTS / "20260616T224121Z"
WITHIN_DIR  = ARTIFACTS / "20260612T233845Z"
WITHIN_PRED = WITHIN_DIR / "predictions_within"
MODELS_DIR  = next((p for p in [base / "webapp/models", base / "models"] if p.exists()), base / "webapp/models")
RAW_SISAL   = base / "data/sisalv3_database_mysql_csv/sisalv3_database_mysql_csv/sisalv3_csv"
OUT         = ARTIFACTS / "revision_2026_11"
FIG         = OUT / "figures"
for d in (OUT, FIG): d.mkdir(parents=True, exist_ok=True)
print("base       :", base, "| exists:", base.exists())
print("models     :", MODELS_DIR, "| exists:", MODELS_DIR.exists())
print("raw SISAL  :", RAW_SISAL, "| exists:", RAW_SISAL.exists())
print("output dir :", OUT)

# ---- Constants (match the manuscript) --------------------------------------------
TARGETS  = ["d18O_measurement", "d13C_measurement", "Mg_Ca_measurement", "Sr_Ca_measurement"]
T_LBL    = {"d18O_measurement": "d18O", "d13C_measurement": "d13C",
            "Mg_Ca_measurement": "Mg/Ca", "Sr_Ca_measurement": "Sr/Ca"}
T_UNIT   = {"d18O_measurement": "permil", "d13C_measurement": "permil",
            "Mg_Ca_measurement": "mmol/mol", "Sr_Ca_measurement": "mmol/mol"}
MODELS   = ["RandomForest", "XGBoost", "CatBoost"]
LOCO_BEST  = {"d18O_measurement": "RandomForest", "d13C_measurement": "CatBoost",
              "Mg_Ca_measurement": "CatBoost", "Sr_Ca_measurement": "RandomForest"}   # Table 1
WCBG_BEST  = {"d18O_measurement": "CatBoost", "d13C_measurement": "CatBoost",
              "Mg_Ca_measurement": "CatBoost", "Sr_Ca_measurement": "XGBoost"}        # Table 2
PUB_LOCO_R2 = {"d18O_measurement": 0.516, "d13C_measurement": 0.182,
               "Mg_Ca_measurement": -0.553, "Sr_Ca_measurement": 0.564}               # LOCO_BEST rows
PUB_WCBG_R2 = {("RandomForest", "d18O_measurement"): 0.971, ("XGBoost", "d18O_measurement"): 0.974,
               ("CatBoost", "d18O_measurement"): 0.977, ("RandomForest", "d13C_measurement"): 0.913,
               ("XGBoost", "d13C_measurement"): 0.906, ("CatBoost", "d13C_measurement"): 0.917,
               ("RandomForest", "Mg_Ca_measurement"): 0.925, ("XGBoost", "Mg_Ca_measurement"): 0.941,
               ("CatBoost", "Mg_Ca_measurement"): 0.958, ("RandomForest", "Sr_Ca_measurement"): 0.981,
               ("XGBoost", "Sr_Ca_measurement"): 0.987, ("CatBoost", "Sr_Ca_measurement"): 0.985}
PUB_LI_RMSE = {"d18O_measurement": 0.469, "d13C_measurement": 0.897,
               "Mg_Ca_measurement": 1.866, "Sr_Ca_measurement": 0.072}                # Table 2, LI rows
NUM_FEATS = ["latitude", "longitude", "elevation", "depth_sample", "interp_age",
             "P_Ca_measurement", "U_Ca_measurement", "Ba_Ca_measurement",
             "age_sin", "age_cos", "log_depth_sample", "precip_d18o_bw02", "abs_latitude",
             *[f"wc_bio{i}" for i in range(1, 20)]]                                  # pipeline CONFIG
NOM_FEATS = ["mineralogy", "monitoring", "hemisphere", "lat_band"]
SEED = 42
REPORT = {}   # everything the final report cell prints

# ---- CorrelationDropper (must exist before any joblib.load) ------------------------
class CorrelationDropper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = float(threshold)
    def fit(self, X, y=None):
        Xa = np.asarray(X, dtype=float)
        self.n_features_in_ = Xa.shape[1]
        if Xa.shape[0] < 4 or Xa.shape[1] < 2:
            self.kept_mask_ = np.ones(self.n_features_in_, dtype=bool)
            self.dropped_indices_ = np.array([], dtype=int); return self
        with np.errstate(invalid="ignore", divide="ignore"):
            corr = np.corrcoef(Xa, rowvar=False)
        corr = np.where(np.isfinite(corr), corr, 0.0)
        upper = np.triu(np.abs(corr), k=1)
        to_drop = set()
        for i in range(corr.shape[0]):
            for j in range(i + 1, corr.shape[1]):
                if upper[i, j] >= self.threshold and j not in to_drop:
                    to_drop.add(j)
        self.kept_mask_ = np.array([k not in to_drop for k in range(self.n_features_in_)], dtype=bool)
        self.dropped_indices_ = np.array(sorted(to_drop), dtype=int)
        return self
    def transform(self, X):
        return np.asarray(X)[:, self.kept_mask_]
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = np.array([f"x{i}" for i in range(self.n_features_in_)])
        return np.asarray(input_features)[self.kept_mask_]

def make_preprocessor(num_feats, nom_feats=NOM_FEATS):
    """Identical to the pipeline's make_column_transformer_factory."""
    num = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", RobustScaler()),
                    ("drop_corr", CorrelationDropper(0.95))])
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=2,
                                          sparse_output=False))])
    return ColumnTransformer([("num", num, list(num_feats)), ("cat", cat, list(nom_feats))],
                             remainder="drop", verbose_feature_names_out=False)

def rmse(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float); return float(np.sqrt(np.mean((a - b) ** 2)))
def r2(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ss = np.sum((a - a.mean()) ** 2); return float(1 - np.sum((a - b) ** 2) / ss) if ss > 0 else np.nan

# ---- Data ------------------------------------------------------------------------
master = pd.read_parquet(ARTIFACTS / "_data" / "master_table_enriched.parquet").reset_index(drop=True)
master["_pos"] = master.groupby("entity_id").cumcount()          # row order inside each entity
_mono = master.groupby("entity_id")["interp_age"].apply(lambda a: bool((a.diff().dropna() >= 0).all()))
print(f"[OK] master_table {master.shape}; entities with age-monotonic row order: {_mono.sum()}/{len(_mono)}")

within_preds = {}
for p in WITHIN_PRED.glob("*.parquet"):
    name, tgt = p.stem.split("_", 1)
    within_preds[(name, tgt)] = pd.read_parquet(p)
print(f"[OK] within-cave prediction files: {len(within_preds)}")

prod = {}
for m in MODELS:
    for t in TARGETS:
        f = MODELS_DIR / f"{m}_{t}.joblib"
        if f.exists():
            prod[(m, t)] = joblib.load(f)
print(f"[OK] production pipelines loaded: {len(prod)}")
def kept_numeric_features(pipe):
    ct = pipe.named_steps["preprocess"]
    cols = [c for n, tr, cs in ct.transformers_ if n == "num" for c in cs]
    dc = ct.named_transformers_["num"].named_steps.get("drop_corr")
    return [c for c, k in zip(cols, dc.kept_mask_) if k] if dc is not None else cols
_feat_check = {k: ("interp_age" in kept_numeric_features(v)) for k, v in prod.items()}
REPORT["interp_age_is_predictor_in_production_models"] = f"{sum(_feat_check.values())}/{len(_feat_check)}"
print("interp_age used as predictor in production models:", REPORT["interp_age_is_predictor_in_production_models"])

## 1 · Inputs and output folders
Reads only what is already on Drive: the saved within-cave predictions, the production models, and the outputs of
the two reviewer-response notebooks in `artifacts/revision_2026_11/`. **Nothing is retrained.** Writes everything
to `artifacts/revision_2026_11/supplement/` (tables as CSV + one Excel workbook, figures as PNG + PDF).

In [ ]:
SUPP = OUT / "supplement"; TABS = SUPP / "tables"; FIGS = SUPP / "figures"
for d in (SUPP, TABS, FIGS): d.mkdir(parents=True, exist_ok=True)

NEEDED = {
    "representativeness site table": OUT / "sisal_site_representativeness.csv",
    "ablation summary": OUT / "ablation_summary.csv",
    **{f"OOF RF tree spread {T_LBL[t]}": OUT / f"oof_rf_treespread_{t}.parquet" for t in TARGETS},
    **{f"hiatus-centred blocks {T_LBL[t]}": OUT / f"hiatus_centred_{WCBG_BEST[t]}_{t}.parquet" for t in TARGETS},
    "LOCO best_params folder": LOCO_DIR / "best_params",
    "within-cave best_params folder": WITHIN_DIR / "best_params",
}
_missing = [k for k, p in NEEDED.items() if not p.exists()]
REPORT["0_inputs_missing"] = _missing or "none"
print("Missing inputs:", _missing or "none")
assert not [k for k in _missing if "best_params" not in k], "Run notebooks 1 and 2 to completion first."

# Colours: model identity uses slots 1-3 of a colour-vision-deficiency-validated categorical palette;
# linear interpolation is a neutral reference (grey, dashed). Text never takes a series colour.
M_COL  = {"RandomForest": "#2a78d6", "XGBoost": "#eb6834", "CatBoost": "#1baf7a"}
M_SHORT = {"RandomForest": "RF", "XGBoost": "XGBoost", "CatBoost": "CatBoost"}
LI_COL, INK, MUTED, GRID = "#52514e", "#0b0b0b", "#8a8984", "#e6e5e1"
T_NICE = {"d18O_measurement": "δ¹⁸O", "d13C_measurement": "δ¹³C", "Mg_Ca_measurement": "Mg/Ca", "Sr_Ca_measurement": "Sr/Ca"}
T_U    = {"d18O_measurement": "‰ VPDB", "d13C_measurement": "‰ VPDB", "Mg_Ca_measurement": "mmol/mol", "Sr_Ca_measurement": "mmol/mol"}
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "xtick.labelsize": 7,
                     "ytick.labelsize": 7, "legend.fontsize": 7.5, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "savefig.facecolor": "white",
                     "figure.facecolor": "white", "font.family": "DejaVu Sans"})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    FILES.append(f"figures/{name}.png / .pdf")
FILES, TABLES = [], {}
def save_table(df, name):
    df.to_csv(TABS / f"{name}.csv", index=False); TABLES[name] = df; FILES.append(f"tables/{name}.csv")

## 2 · Rebuild the 46 held-out blocks (same code as notebook 1)

In [ ]:
_midx = master.set_index("sample_id")
_ent_cache = {eid: g.sort_values("_pos") for eid, g in master.groupby("entity_id")}

def build_blocks(model, tgt):
    pf = within_preds[(model, tgt)][["fold", "entity_id", "sample_id", "y_true", "y_pred"]].copy()
    pf["pos"] = _midx.loc[pf["sample_id"].values, "_pos"].to_numpy()
    brows, srows = [], []
    for fold, g in pf.groupby("fold"):
        g = g.sort_values("pos"); ent = int(g["entity_id"].iloc[0]); e = _ent_cache[ent]
        y = e[tgt].to_numpy(float); age = e["interp_age"].to_numpy(float); dep = e["depth_sample"].to_numpy(float)
        lo, hi = int(g["pos"].min()), int(g["pos"].max()); n = len(g)
        inb = np.zeros(len(e), bool); inb[lo:hi + 1] = True
        sd_ent = float(np.std(y[~inb], ddof=1))
        yb = g["y_true"].to_numpy(float); yp = g["y_pred"].to_numpy(float)
        edge = lo == 0 or hi == len(e) - 1
        if not edge:
            a0, a1, y0, y1 = age[lo - 1], age[hi + 1], y[lo - 1], y[hi + 1]
            li_idx = np.linspace(y0, y1, n + 2)[1:-1]
            li_age = np.interp(age[lo:hi + 1], [a0, a1], [y0, y1]) if a1 > a0 else li_idx
            duration = a1 - a0; step = abs(y1 - y0) / sd_ent
        else:
            li_idx = li_age = np.full(n, np.nan); duration = age[hi] - age[lo]; step = np.nan
        brows.append(dict(model=model, target=tgt, fold=fold, entity_id=ent, site_name=e["site_name"].iloc[0], n=n,
                          edge=edge, duration_yr=duration, sd_ent=sd_ent, pos_lo=lo, pos_hi=hi, step=step,
                          departure=rmse(yb, li_age) / sd_ent if not edge else np.nan,
                          rmse_model=rmse(yb, yp), rmse_li_age=rmse(yb, li_age) if not edge else np.nan))
        srows.append(pd.DataFrame(dict(model=model, target=tgt, fold=fold, entity_id=ent, sample_id=g["sample_id"].values,
                                       age=age[lo:hi + 1], depth=dep[lo:hi + 1], y_true=yb, y_pred=yp,
                                       li_age=li_age, li_idx=li_idx, sd_ent=sd_ent)))
    bt = pd.DataFrame(brows)
    bt["nrmse_model"] = bt.rmse_model / bt.sd_ent; bt["nrmse_li_age"] = bt.rmse_li_age / bt.sd_ent
    return bt, pd.concat(srows, ignore_index=True)

BT, ST = {}, {}
for (m, t) in sorted(within_preds):
    BT[(m, t)], ST[(m, t)] = build_blocks(m, t)
_chk = {f"{m}/{T_LBL[t]}": (round(r2(s.y_true, s.y_pred), 3), PUB_WCBG_R2.get((m, t))) for (m, t), s in ST.items()}
REPORT["1_wcbg_reproduction (rebuilt, published)"] = _chk
assert all(a == b for a, b in _chk.values()), "Rebuilt blocks do not reproduce Table 2 - stop and report."
print("[OK] all 12 within-cave pooled R² reproduce Table 2")

## 3 · Check A — the linear-interpolation (LI) row of Table 2
Table 2's LI **R²** came from one pipeline cell (interpolation by sample order, all 2,098 samples) and its
**RMSE / IQR / MAE / MdAE** and the Wilcoxon/TOST tests from another (interpolation against depth). Depth is
missing for every sample of one stalagmite, so the depth version silently drops that entity's block (2,098 → 2,033).
This cell reproduces the published numbers, identifies the dropped samples, and recomputes a single consistent LI
row by interpolating in **age** (available for every sample), with the same statistics as the paper.

In [ ]:
from scipy.stats import wilcoxon

def li_depth_published(model, tgt):
    """Exact re-implementation of the pipeline's Table-2 LI cell (interpolation against depth)."""
    pf = within_preds[(model, tgt)]; pred = pf.set_index("sample_id")["y_pred"]
    out, per = [], []
    for eid, g in pf.groupby("entity_id"):
        ent = master[master.entity_id == eid].sort_values("depth_sample").reset_index(drop=True)
        held = ent.sample_id.isin(g.sample_id).to_numpy()
        if held.sum() < 5: continue
        y = ent[tgt].to_numpy(float); x = ent["depth_sample"].to_numpy(float)
        anch = (~held) & np.isfinite(y) & np.isfinite(x)
        if anch.sum() < 2:
            per.append(dict(entity_id=eid, n=int(held.sum()), dropped=True)); continue
        yt = y[held]; ym = pred.reindex(ent.loc[held, "sample_id"]).to_numpy(float)
        yl = np.interp(x[held], x[anch], y[anch])
        ok = np.isfinite(yt) & np.isfinite(ym) & np.isfinite(yl)
        out.append(pd.DataFrame(dict(entity_id=eid, y_true=yt[ok], y_ml=ym[ok], y_li=yl[ok])))
        per.append(dict(entity_id=eid, n=int(held.sum()), dropped=bool(ok.sum() < held.sum())))
    return pd.concat(out, ignore_index=True), pd.DataFrame(per)

def pooled_stats(yt, yp, folds):
    e = np.abs(yt - yp); fr = pd.Series((yt - yp) ** 2).groupby(np.asarray(folds)).mean() ** 0.5
    return dict(n=len(yt), R2=round(r2(yt, yp), 3), RMSE=round(rmse(yt, yp), 3),
                RMSE_IQR=f"{fr.quantile(.25):.2f}–{fr.quantile(.75):.2f}", MAE=round(float(e.mean()), 3),
                MdAE=round(float(np.median(e)), 3))

def compare_tests(per_fold_ml, per_fold_li, margin=0.20):
    d = per_fold_ml - per_fold_li
    p = wilcoxon(per_fold_ml, per_fold_li).pvalue
    dn = ((per_fold_li - per_fold_ml) / per_fold_li.replace(0, np.nan)).dropna()
    pn = wilcoxon(dn).pvalue
    bound = margin * per_fold_li.median()
    p_tost = max(wilcoxon(d + bound, alternative="greater").pvalue, wilcoxon(d - bound, alternative="less").pvalue)
    return dict(n_blocks=len(d), model_wins=f"{int((per_fold_ml < per_fold_li).sum())}/{len(d)}",
                wilcoxon_p=round(p, 4), wilcoxon_p_bonf=round(min(1, 4 * p), 4),
                normalised_p_bonf=round(min(1, 4 * pn), 4), tost_p_bonf=round(min(1, 4 * p_tost), 4),
                equivalent_at_20pct=bool(min(1, 4 * p_tost) <= 0.05))

li_rows, test_rows, drop_rows = [], [], []
for t in TARGETS:
    m = WCBG_BEST[t]; st = ST[(m, t)]
    li_rows.append(dict(proxy=T_NICE[t], method="Model (" + M_SHORT[m] + "), all samples [Table 2 model row]",
                        **pooled_stats(st.y_true.to_numpy(), st.y_pred.to_numpy(), st.fold)))
    st_i = st.dropna(subset=["li_idx"])
    li_rows.append(dict(proxy=T_NICE[t], method="LI by sample order [published R² source]",
                        **pooled_stats(st_i.y_true.to_numpy(), st_i.li_idx.to_numpy(), st_i.fold)))
    dp, per = li_depth_published(m, t)
    fold_of = dict(zip(st.entity_id, st.fold))
    li_rows.append(dict(proxy=T_NICE[t], method="LI against depth [published RMSE/MAE/MdAE/tests source]",
                        **pooled_stats(dp.y_true.to_numpy(), dp.y_li.to_numpy(), dp.entity_id.map(fold_of))))
    st_a = st.dropna(subset=["li_age"])
    li_rows.append(dict(proxy=T_NICE[t], method="LI against age, all samples [PROPOSED single LI row]",
                        **pooled_stats(st_a.y_true.to_numpy(), st_a.li_age.to_numpy(), st_a.fold)))
    for _, r in per[per.dropped].iterrows():
        drop_rows.append(dict(proxy=T_NICE[t], entity_id=r.entity_id, site=_ent_cache[r.entity_id].site_name.iloc[0],
                              held_out_samples=r.n,
                              depth_missing_in_entity=int(_ent_cache[r.entity_id].depth_sample.isna().sum()),
                              entity_samples=len(_ent_cache[r.entity_id])))
    # per-block tests: published (depth) and proposed (age)
    g = dp.groupby("entity_id")
    f_ml = g.apply(lambda x: rmse(x.y_true, x.y_ml)); f_li = g.apply(lambda x: rmse(x.y_true, x.y_li))
    test_rows.append(dict(proxy=T_NICE[t], model=M_SHORT[m], li="depth [published]", **compare_tests(f_ml, f_li)))
    bt = BT[(m, t)].dropna(subset=["rmse_li_age"])
    test_rows.append(dict(proxy=T_NICE[t], model=M_SHORT[m], li="age [proposed]",
                          **compare_tests(bt.rmse_model.reset_index(drop=True), bt.rmse_li_age.reset_index(drop=True))))
LI_TABLE, LI_TESTS, LI_DROPPED = pd.DataFrame(li_rows), pd.DataFrame(test_rows), pd.DataFrame(drop_rows)
REPORT["2a_table2_LI_versions"] = LI_TABLE
REPORT["2b_model_vs_LI_tests"] = LI_TESTS
REPORT["2c_samples_dropped_by_depth_LI"] = LI_DROPPED if len(LI_DROPPED) else "none"
print(LI_TABLE.to_string(index=False)); print(LI_TESTS.to_string(index=False)); print(LI_DROPPED.to_string(index=False))

# LI R² for every proxy with the proposed method, to restate the comparisons in Section 3.2
_li_r2 = {T_NICE[t]: round(r2(ST[(WCBG_BEST[t], t)].dropna(subset=["li_age"]).y_true,
                               ST[(WCBG_BEST[t], t)].dropna(subset=["li_age"]).li_age), 3) for t in TARGETS}
REPORT["2d_LI_age_R2_by_proxy"] = _li_r2

## 4 · Check B — tuned hyper-parameters in the LOCO and within-cave runs
The production models used in R2 carry the hyper-parameters of the **within-cave** run, while Table 1 comes from the
**LOCO** run. This compares the two runs' tuned parameters for every model/proxy.

In [ ]:
def _load_bp(folder, m, t):
    p = folder / f"{m}_{t}.json"
    if not p.exists(): return None
    d = json.loads(p.read_text()); return d.get("best_params", d)
hp_rows = []
for m in MODELS:
    for t in TARGETS:
        a, b = _load_bp(LOCO_DIR / "best_params", m, t), _load_bp(WITHIN_DIR / "best_params", m, t)
        if a is None or b is None:
            hp_rows.append(dict(model=m, target=T_LBL[t], identical=None, differences="file missing")); continue
        diff = {k: (a.get(k), b.get(k)) for k in sorted(set(a) | set(b)) if a.get(k) != b.get(k)}
        hp_rows.append(dict(model=m, target=T_LBL[t], identical=not diff,
                            differences="; ".join(f"{k.split('__')[-1]}: LOCO={v[0]} vs within={v[1]}" for k, v in diff.items()) or "-"))
REPORT["3_tuned_params_LOCO_vs_within_run"] = pd.DataFrame(hp_rows)
print(REPORT["3_tuned_params_LOCO_vs_within_run"].to_string(index=False))

## 5 · Supplementary tables

In [ ]:
# ---- S-table: representativeness of the 41 caves -------------------------------------------------
per = pd.read_csv(OUT / "sisal_site_representativeness.csv", index_col=0)
ref = per[per.has_isotopes.astype(bool) & ~per.cohort.astype(bool)]; coh = per[per.cohort.astype(bool)]
LABEL = {"latitude": "Latitude (°)", "abs_lat": "Absolute latitude (°)", "elevation": "Elevation (m)",
         "age_min": "Youngest age (yr BP)", "age_max": "Oldest age (yr BP)", "age_span": "Record span (yr)",
         "d18O_median": "Median δ¹⁸O (‰)", "d13C_median": "Median δ¹³C (‰)",
         "MAT_C": "Mean annual temperature (°C)", "MAP_mm": "Mean annual precipitation (mm)"}
rows = []
for v, lab in LABEL.items():
    if v not in per: continue
    a, b = coh[v].dropna(), ref[v].dropna(); ks = stats.ks_2samp(a, b)
    rows.append({"Variable": lab, "Cohort median (IQR)": f"{a.median():.1f} ({a.quantile(.25):.1f} to {a.quantile(.75):.1f})",
                 "Other sites median (IQR)": f"{b.median():.1f} ({b.quantile(.25):.1f} to {b.quantile(.75):.1f})",
                 "KS D": round(ks.statistic, 2), "KS p": round(ks.pvalue, 3),
                 "Other sites within cohort range (%)": round(100 * b.between(a.min(), a.max()).mean(), 1)})
for v, lab in [("lat_band", "Latitude band"), ("hemisphere", "Hemisphere"), ("continent", "Continent")]:
    tab = pd.crosstab(per.loc[per.has_isotopes.astype(bool), v], per.loc[per.has_isotopes.astype(bool), "cohort"].astype(bool))
    p = stats.chi2_contingency(tab.values)[1]
    for lvl, r in tab.iterrows():
        rows.append({"Variable": f"{lab}: {lvl}", "Cohort median (IQR)": f"{int(r.get(True, 0))} sites ({100*r.get(True, 0)/tab[True].sum():.0f}%)",
                     "Other sites median (IQR)": f"{int(r.get(False, 0))} sites ({100*r.get(False, 0)/tab[False].sum():.0f}%)",
                     "KS D": None, "KS p": round(p, 3), "Other sites within cohort range (%)": None})
S_REP = pd.DataFrame(rows).rename(columns={"KS p": "p (KS or χ²)"})
save_table(S_REP, "S_representativeness")
counts = dict(sites=len(per), with_MgCa=int(per.has_MgCa.fillna(False).astype(bool).sum()),
              with_SrCa=int(per.has_SrCa.fillna(False).astype(bool).sum()),
              with_all4_same_samples=int((per.n_all4_samples > 0).sum()), cohort=len(coh))
REPORT["4a_representativeness_counts"] = counts

# ---- S-table: feature ablation ---------------------------------------------------------------------
ab = pd.read_csv(OUT / "ablation_summary.csv")
VNAME = {"full": "All features (re-run)", "no_harmonics": "Without 21-kyr sin/cos", "no_interp_age": "Without interpolated age",
         "no_age": "Without any age feature", "p19": "Period 19 kyr", "p23": "Period 23 kyr", "p41": "Period 41 kyr", "p100": "Period 100 kyr"}
ORD = list(VNAME)
ab["vo"] = ab.variant.map({v: i for i, v in enumerate(ORD)})
S_ABL = (ab.sort_values(["protocol", "target", "vo"])
           .assign(Variant=lambda d: d.variant.map(VNAME), Protocol=lambda d: d.protocol.map({"LOCO": "Leave-one-cave-out", "WCBG": "Within-cave block gap"}))
           [["Protocol", "target", "model", "Variant", "n_folds", "pooled_r2", "pooled_rmse", "delta_r2_vs_full",
             "folds_worse_than_full", "wilcoxon_p_fold_rmse"]]
           .rename(columns={"target": "Proxy", "model": "Model", "n_folds": "Folds", "pooled_r2": "Pooled R²",
                            "pooled_rmse": "Pooled RMSE", "delta_r2_vs_full": "ΔR² vs all features",
                            "folds_worse_than_full": "Folds worse", "wilcoxon_p_fold_rmse": "Wilcoxon p (fold RMSE)"}))
save_table(S_ABL, "S_feature_ablation")

# ---- S-table: gap duration ------------------------------------------------------------------------
def partial_spearman(x, y, z):
    rx, ry, rz = (stats.rankdata(v) for v in (x, y, z))
    ex = rx - np.polyval(np.polyfit(rz, rx, 1), rz); ey = ry - np.polyval(np.polyfit(rz, ry, 1), rz)
    return stats.pearsonr(ex, ey)
rows = []
for t in TARGETS:
    for m in MODELS:
        bt = BT[(m, t)].dropna(subset=["nrmse_li_age"]); bt = bt[bt.duration_yr > 0]
        r0 = stats.spearmanr(bt.duration_yr, bt.rmse_model); r1 = stats.spearmanr(bt.duration_yr, bt.nrmse_model)
        rp = partial_spearman(bt.duration_yr, bt.nrmse_model, bt.n)
        rd = stats.spearmanr(bt.duration_yr, bt.nrmse_model - bt.nrmse_li_age)
        rows.append({"Proxy": T_NICE[t], "Model": M_SHORT[m], "Blocks": len(bt),
                     "ρ duration–RMSE": round(r0[0], 2), "p": round(r0[1], 3),
                     "ρ duration–nRMSE": round(r1[0], 2), "p ": round(r1[1], 3),
                     "partial ρ (| samples in gap)": round(rp[0], 2), "p  ": round(rp[1], 3),
                     "ρ duration–(model − LI nRMSE)": round(rd[0], 2), "p   ": round(rd[1], 3)})
    bt = BT[("RandomForest", t)].dropna(subset=["nrmse_li_age"]); bt = bt[bt.duration_yr > 0]
    r = stats.spearmanr(bt.duration_yr, bt.nrmse_li_age)
    rows.append({"Proxy": T_NICE[t], "Model": "Linear interpolation", "Blocks": len(bt), "ρ duration–nRMSE": round(r[0], 2), "p ": round(r[1], 3)})
S_DUR = pd.DataFrame(rows); save_table(S_DUR, "S_gap_duration")

# ---- S-table: abrupt change (departure / step tertiles) -------------------------------------------
rows = []
for t in TARGETS:
    m = WCBG_BEST[t]; bt = BT[(m, t)].dropna(subset=["departure"])
    for var, nm in [("departure", "Departure from linear bridge"), ("step", "Step between bounding samples")]:
        b2 = bt.assign(tert=pd.qcut(bt[var], 3, labels=["Low", "Middle", "High"]))
        for tert, g in b2.groupby("tert", observed=True):
            w = stats.wilcoxon(g.nrmse_model, g.nrmse_li_age).pvalue
            rows.append({"Proxy": T_NICE[t], "Model": M_SHORT[m], "Stratified by": nm, "Third": tert, "Blocks": len(g),
                         "Range (× record SD)": f"{g[var].min():.2f}–{g[var].max():.2f}",
                         "Median nRMSE model": round(g.nrmse_model.median(), 2), "Median nRMSE LI": round(g.nrmse_li_age.median(), 2),
                         "Model better (%)": round(100 * (g.nrmse_model < g.nrmse_li_age).mean()), "Wilcoxon p": round(w, 3)})
S_ABR = pd.DataFrame(rows); save_table(S_ABR, "S_abrupt_change")

# ---- S-table: hiatus-centred blocks ---------------------------------------------------------------
HC = pd.concat([pd.read_parquet(OUT / f"hiatus_centred_{WCBG_BEST[t]}_{t}.parquet") for t in TARGETS], ignore_index=True)
S_HIA = (HC.assign(ratio=lambda d: d.step_at_hiatus / d.step_random_median)
           [["target", "model", "site_name", "entity_id", "hiatus_depth", "n", "duration_yr", "step_at_hiatus",
             "step_random_median", "nrmse_model", "nrmse_li_age"]]
           .rename(columns={"target": "Proxy", "model": "Model", "site_name": "Site", "entity_id": "Entity",
                            "hiatus_depth": "Hiatus depth (mm)", "n": "Block samples", "duration_yr": "Interval between bounding samples (yr)",
                            "step_at_hiatus": "Step at hiatus (× SD)", "step_random_median": "Median step at random positions (× SD)",
                            "nrmse_model": "nRMSE model", "nrmse_li_age": "nRMSE LI"}).round(3))
save_table(S_HIA, "S_hiatus_centred_blocks")

# ---- S-table: interval coverage -------------------------------------------------------------------
def conformal_half(df, col, alpha=0.05):
    r = np.abs(df.y_true.to_numpy() - df[col].to_numpy()); f = df.fold.to_numpy(); h = np.empty(len(df))
    for k in np.unique(f):
        o = r[f != k]; h[f == k] = np.quantile(o, min(1, (1 - alpha) * (1 + 1 / len(o))), method="higher")
    return h
X_ALL = master.drop(columns=TARGETS + ["_pos"]); SID2ROW = pd.Series(master.index.to_numpy(), index=master.sample_id.to_numpy())
rows = []
for t in TARGETS:
    st = ST[("RandomForest", t)]; pipe = prod[("RandomForest", t)]
    Xt = pipe.named_steps["preprocess"].transform(X_ALL.loc[SID2ROW.loc[st.sample_id].to_numpy()])
    P = np.array([tr.predict(Xt) for tr in pipe.named_steps["estimator"].estimators_]); mu, sd = P.mean(0), P.std(0)
    rows.append({"Proxy": T_NICE[t], "Interval": "RF tree spread, production model (in-sample; as submitted)",
                 "Coverage (%)": round(100 * np.mean(np.abs(st.y_true - mu) <= 1.96 * sd), 1), "Mean width": round(float(np.mean(3.92 * sd)), 3)})
    o = pd.read_parquet(OUT / f"oof_rf_treespread_{t}.parquet")
    rows.append({"Proxy": T_NICE[t], "Interval": "RF tree spread, fold models (out-of-fold)",
                 "Coverage (%)": round(100 * np.mean(np.abs(o.y_true - o.mu) <= 1.96 * o.sd), 1), "Mean width": round(float(np.mean(3.92 * o.sd)), 3),
                 "Mean tree SD / RMSE": round(float(o.sd.mean()) / rmse(o.y_true, o.mu), 2)})
    for m in [WCBG_BEST[t], "LI"]:
        src = ST[(WCBG_BEST[t], t)].dropna(subset=["li_age"]); col = "li_age" if m == "LI" else "y_pred"
        h = conformal_half(src, col); ins = np.abs(src.y_true - src[col]) <= h
        pe = pd.Series(ins.to_numpy()).groupby(src.entity_id.to_numpy()).mean()
        rows.append({"Proxy": T_NICE[t], "Interval": f"Cross-conformal around {'linear interpolation' if m == 'LI' else M_SHORT[m]}",
                     "Coverage (%)": round(100 * ins.mean(), 1), "Mean width": round(float(np.mean(2 * h)), 3),
                     "Records with coverage ≥ 90 (%)": round(100 * (pe >= 0.90).mean()),
                     "Lowest record coverage (%)": round(100 * pe.min(), 1)})
S_COV = pd.DataFrame(rows); save_table(S_COV, "S_interval_coverage")

# ---- S-table: LI row and tests (Check A) ------------------------------------------------------------
save_table(LI_TABLE, "S_check_table2_LI_versions"); save_table(LI_TESTS, "S_check_model_vs_LI_tests")
save_table(REPORT["3_tuned_params_LOCO_vs_within_run"], "S_check_tuned_params")

_pip("openpyxl")
with pd.ExcelWriter(SUPP / "Supplementary_Tables.xlsx") as xw:
    for k, v in TABLES.items(): v.to_excel(xw, sheet_name=k[:31], index=False)
FILES.append("Supplementary_Tables.xlsx")
print("[OK] tables written:", list(TABLES))

## 6 · Supplementary figures

In [ ]:
# ---- Map ---------------------------------------------------------------------------------------------
def _map(use_cartopy):
    if use_cartopy:
        _pip("cartopy"); import cartopy.crs as ccrs, cartopy.feature as cfeature
        fig = plt.figure(figsize=(7.2, 3.6)); ax = plt.axes(projection=ccrs.Robinson()); ax.set_global()
        ax.add_feature(cfeature.LAND, facecolor="#efeeea", edgecolor="none"); ax.coastlines(linewidth=0.3, color=MUTED)
        kw = dict(transform=ccrs.PlateCarree())
    else:
        fig, ax = plt.subplots(figsize=(7.2, 3.6)); kw = {}
        ax.set_xlim(-180, 180); ax.set_ylim(-60, 80); ax.set_xlabel("Longitude (°)"); ax.set_ylabel("Latitude (°)")
    ax.scatter(ref.longitude, ref.latitude, s=7, c="#b4b2a9", linewidths=0, label=f"Other SISALv3 sites (n = {len(ref)})", **kw)
    ax.scatter(coh.longitude, coh.latitude, s=22, c=M_COL["RandomForest"], edgecolor="white", linewidth=0.6,
               label=f"Study cohort (n = {len(coh)})", **kw)
    ax.legend(loc="lower left", frameon=False); save_fig(fig, "S_fig_cohort_map"); return fig
try: fig = _map(True)
except Exception as ex:
    print("cartopy map failed, plain lon/lat map instead:", ex); plt.close("all"); FILES[:] = [f for f in FILES if "cohort_map" not in f]; fig = _map(False)
plt.show()

def plain_log(ax, x=True, y=True):
    from matplotlib.ticker import LogLocator, FuncFormatter, NullFormatter
    f = FuncFormatter(lambda v, _: f"{v:g}")
    for on, axis in [(x, ax.xaxis), (y, ax.yaxis)]:
        if on:
            axis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0), numticks=8))
            axis.set_major_formatter(f); axis.set_minor_formatter(NullFormatter())
# ---- Error vs gap duration ---------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.3), sharey=False)
for ax, t in zip(axes, TARGETS):
    m = WCBG_BEST[t]; bt = BT[(m, t)].dropna(subset=["nrmse_li_age"]); bt = bt[bt.duration_yr > 0]
    ax.scatter(bt.duration_yr, bt.nrmse_li_age, s=12, facecolors="none", edgecolors=LI_COL, linewidths=0.8, label="Linear interpolation")
    ax.scatter(bt.duration_yr, bt.nrmse_model, s=12, c=M_COL[m], edgecolors="white", linewidths=0.4, label="Best within-cave model (named in panel title)")
    rm = stats.spearmanr(bt.duration_yr, bt.nrmse_model); rl = stats.spearmanr(bt.duration_yr, bt.nrmse_li_age)
    ax.text(0.03, 0.97, f"model ρ = {rm[0]:.2f} (p = {rm[1]:.2f})\nLI ρ = {rl[0]:.2f} (p = {rl[1]:.2f})",
            transform=ax.transAxes, va="top", fontsize=6.5, color=INK,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1))
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(f"{T_NICE[t]} · {M_SHORT[m]}", color=INK); ax.set_xlabel("Gap duration (years)")
    plain_log(ax, x=False, y=True)
axes[0].set_ylabel("nRMSE (RMSE / record SD)")
h, l = axes[0].get_legend_handles_labels(); fig.legend(h, l, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.08))
fig.tight_layout(); save_fig(fig, "S_fig_error_vs_duration"); plt.show()

# ---- Error vs departure from a linear bridge ---------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.3))
for ax, t in zip(axes, TARGETS):
    m = WCBG_BEST[t]; bt = BT[(m, t)].dropna(subset=["departure"])
    ax.scatter(bt.departure, bt.nrmse_model, s=12, c=M_COL[m], edgecolors="white", linewidths=0.4, label="Best within-cave model (named in panel title)")
    xl = [bt.departure.min() * 0.8, bt.departure.max() * 1.25]
    yl = [min(bt.nrmse_model.min(), xl[0]) * 0.9, max(bt.nrmse_model.max(), xl[1]) * 1.1]
    lim = [min(xl[0], yl[0]), max(xl[1], yl[1])]
    ax.plot(lim, lim, color=LI_COL, lw=1, ls="--", label="Linear interpolation (= departure)")
    rr = stats.spearmanr(bt.departure, bt.nrmse_model)
    ax.text(0.03, 0.97, f"ρ = {rr[0]:.2f}", transform=ax.transAxes, va="top", fontsize=6.5, color=INK)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(xl); ax.set_ylim(yl); ax.set_title(f"{T_NICE[t]} · {M_SHORT[m]}", color=INK)
    ax.set_xlabel("Departure (× record SD)")
    plain_log(ax)
axes[0].set_ylabel("nRMSE of model")
h, l = axes[0].get_legend_handles_labels(); fig.legend(h, l, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.08))
fig.tight_layout(); save_fig(fig, "S_fig_error_vs_departure"); plt.show()

# ---- Hiatus-centred blocks: model vs LI, per block ---------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.4), sharey=True)
for ax, t in zip(axes, TARGETS):
    m = WCBG_BEST[t]; g = HC[HC.target == T_LBL[t]]
    for _, r in g.iterrows():
        ax.plot([0, 1], [r.nrmse_li_age, r.nrmse_model], color="#c9c8c1", lw=0.8, zorder=1)
    ax.scatter(np.zeros(len(g)), g.nrmse_li_age, s=14, facecolors="none", edgecolors=LI_COL, linewidths=0.8, zorder=2)
    ax.scatter(np.ones(len(g)), g.nrmse_model, s=14, c=M_COL[m], edgecolors="white", linewidths=0.4, zorder=2)
    rnd = BT[(m, t)].dropna(subset=["nrmse_li_age"]).nrmse_model.median()
    ax.axhline(rnd, color=MUTED, lw=0.8, ls=":")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["LI", M_SHORT[m]]); ax.set_xlim(-0.4, 1.6)
    ax.set_title(f"{T_NICE[t]} (n = {len(g)})", color=INK); ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Normalised RMSE")
from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([], [], color="#c9c8c1", lw=0.8, label="Same hiatus-centred block"),
                    Line2D([], [], color=MUTED, lw=0.8, ls=":", label="Model median, randomly placed blocks")],
           loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.08))
fig.tight_layout(); save_fig(fig, "S_fig_hiatus_blocks"); plt.show()

# ---- Feature ablation: ΔR² vs all features ----------------------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(7.2, 3.8), sharey="row")
for i, proto in enumerate(["LOCO", "WCBG"]):
    for j, t in enumerate(TARGETS):
        ax = axes[i, j]; d = ab[(ab.protocol == proto) & (ab.target == T_LBL[t]) & (ab.variant != "full")].sort_values("vo")
        ylab = [VNAME[v].replace("Without ", "– ").replace("Period ", "") for v in d.variant]
        ax.barh(range(len(d)), d.delta_r2_vs_full, color="#8f8e88", height=0.6)
        ax.axvline(0, color=INK, lw=0.8); ax.set_yticks(range(len(d))); ax.set_yticklabels(ylab); ax.invert_yaxis()
        ax.grid(axis="y", visible=False)
        if i == 0: ax.set_title(T_NICE[t], color=INK)
        if j == 0: ax.set_ylabel("Leave-one-cave-out" if proto == "LOCO" else "Within-cave")
        ax.set_xlabel("ΔR² vs all features" if i == 1 else "")
fig.tight_layout(); save_fig(fig, "S_fig_feature_ablation"); plt.show()

# ---- Coverage --------------------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.3), sharey=True)
def short(x):
    if "production" in x: return "RF\nin-samp."
    if "fold models" in x: return "RF\nOOF"
    return "Conf.\nLI" if "interpolation" in x else "Conf.\nmodel"
for ax, t in zip(axes, TARGETS):
    d = S_COV[S_COV.Proxy == T_NICE[t]].reset_index(drop=True)
    labs = [short(x) for x in d.Interval]
    ax.bar(range(len(d)), d["Coverage (%)"], color="#8f8e88", width=0.62)
    for k, v in enumerate(d["Coverage (%)"]): ax.text(k, v + 1.5, f"{v:.0f}", ha="center", fontsize=6.5, color=INK)
    ax.axhline(95, color=INK, lw=0.8, ls="--"); ax.set_ylim(0, 108); ax.set_xticks(range(len(d)))
    ax.set_xticklabels(labs, fontsize=6); ax.set_title(T_NICE[t], color=INK); ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Coverage of nominal 95% interval (%)")
fig.tight_layout(); save_fig(fig, "S_fig_interval_coverage"); plt.show()

## 7 · Revised Figure 6
The submitted Figure 6 showed, for each model and proxy, the held-out block with the **highest** R² and bands from
in-sample production models. The revised figure shows the block with the **median** normalised error for each
model–proxy pair (a typical case, not the best one), plotted against age with the surrounding measured record,
linear interpolation for comparison, and the cross-conformal 95% interval around the model prediction.

In [ ]:
CONTEXT = 25          # measured samples shown either side of the gap
fig, axes = plt.subplots(3, 4, figsize=(7.2, 6.4))
FIG6_ROWS = []
for i, m in enumerate(MODELS):
    for j, t in enumerate(TARGETS):
        ax = axes[i, j]; bt = BT[(m, t)].dropna(subset=["nrmse_li_age"]).reset_index(drop=True)
        k = int((bt.nrmse_model - bt.nrmse_model.median()).abs().idxmin()); b = bt.loc[k]
        st = ST[(m, t)].dropna(subset=["li_age"]).copy(); st["half"] = conformal_half(st, "y_pred")
        s = st[st.fold == b.fold].sort_values("age")
        e = _ent_cache[b.entity_id]; lo = max(0, b.pos_lo - CONTEXT); hi = min(len(e), b.pos_hi + CONTEXT + 1)
        ctx = e.iloc[lo:hi]; ka = lambda a: np.asarray(a) / 1000
        ax.plot(ka(ctx.interp_age), ctx[t], color="#c9c8c1", lw=0.8, zorder=1)
        ax.scatter(ka(ctx.interp_age), ctx[t], s=3, color="#b4b2a9", zorder=1)
        ax.fill_between(ka(s.age), s.y_pred - s.half, s.y_pred + s.half, color=M_COL[m], alpha=0.18, lw=0, zorder=2)
        ax.plot(ka(s.age), s.li_age, color=LI_COL, lw=1, ls="--", zorder=3)
        ax.plot(ka(s.age), s.y_true, color=INK, lw=1.1, zorder=4)
        ax.plot(ka(s.age), s.y_pred, color=M_COL[m], lw=1.4, zorder=5)
        ax.set_title((f"{T_NICE[t]} ({T_U[t]})\n" if i == 0 else "") + e.site_name.iloc[0], color=INK, fontsize=7.5)
        ax.text(0.02, 0.03, f"RMSE {M_SHORT[m]} {b.rmse_model:.3g} · LI {b.rmse_li_age:.3g}", transform=ax.transAxes,
                fontsize=6, color=INK, va="bottom", bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1))
        if j == 0: ax.set_ylabel(M_SHORT[m], fontsize=9, color=INK)
        if i == 2: ax.set_xlabel("Age (ka BP)")
        FIG6_ROWS.append(dict(model=M_SHORT[m], proxy=T_NICE[t], site=e.site_name.iloc[0], entity_id=int(b.entity_id),
                              block_samples=int(b.n), duration_yr=round(float(b.duration_yr)), rmse_model=round(b.rmse_model, 4),
                              rmse_LI=round(b.rmse_li_age, 4), nrmse_model=round(b.nrmse_model, 3),
                              median_nrmse_all_blocks=round(bt.nrmse_model.median(), 3),
                              band_coverage_in_block_pct=round(100 * np.mean(np.abs(s.y_true - s.y_pred) <= s.half), 1)))
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
handles = [Line2D([], [], color=INK, lw=1.1, label="Measured (held out)"),
           Line2D([], [], color="#c9c8c1", lw=0.8, marker="o", ms=2, label="Measured (training, context)"),
           *[Line2D([], [], color=M_COL[m], lw=1.4, label=f"{M_SHORT[m]} prediction") for m in MODELS],
           Line2D([], [], color=LI_COL, lw=1, ls="--", label="Linear interpolation"),
           Patch(color="#9b9a94", alpha=0.35, label="95% cross-conformal interval (model colour)")]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.04))
fig.tight_layout(rect=(0, 0.04, 1, 1)); save_fig(fig, "Figure6_revised"); plt.show()
REPORT["5_figure6_panels"] = pd.DataFrame(FIG6_ROWS)
save_table(REPORT["5_figure6_panels"], "Figure6_panel_details")

## 8 · Export the exact within-cave held-out blocks (for the public code)
The original splitter seeded block positions with Python's `hash()`, which changes between sessions, so re-running it
could not regenerate Table 2's blocks. This writes the blocks actually used (`wcbg_heldout_blocks.csv`) and checks
that all 12 prediction files used the same blocks. The file is in the repository (`data/wcbg_heldout_blocks.csv`), and
the pipeline notebooks pass it to `BlockedWithinCaveSplitter(fixed_blocks=...)`, so Table 2 is reproduced exactly (see R4).

In [ ]:
_blk = {k: v[["entity_id", "fold", "sample_id"]].sort_values(["entity_id", "sample_id"]).reset_index(drop=True)
        for k, v in within_preds.items()}
_ref = _blk[next(iter(_blk))][["entity_id", "sample_id"]]
_same = {f"{m}/{T_LBL[t]}": bool(v[["entity_id", "sample_id"]].equals(_ref)) for (m, t), v in _blk.items()}
REPORT["6_identical_blocks_across_12_files"] = _same
blocks = (within_preds[next(iter(within_preds))][["entity_id", "sample_id"]]
          .assign(position_in_entity=lambda d: _midx.loc[d.sample_id, "_pos"].to_numpy(),
                  interp_age=lambda d: _midx.loc[d.sample_id, "interp_age"].to_numpy())
          .sort_values(["entity_id", "position_in_entity"]))
blocks.to_csv(SUPP / "wcbg_heldout_blocks.csv", index=False); FILES.append("wcbg_heldout_blocks.csv")
print("identical blocks in all files:", all(_same.values()), "| rows:", len(blocks), "| entities:", blocks.entity_id.nunique())

## 9 · Check C — hyper-parameter search settings recorded
Reads what each saved tuning file recorded (number of halving rounds for the tree ensembles; whether a deep-learning
architecture was reused from δ18O) and the run configuration, documenting the tuning described in Section 2.5.

In [ ]:
rows_c = []
for run, folder in [("LOCO", LOCO_DIR / "best_params"), ("within", WITHIN_DIR / "best_params")]:
    for p in sorted(folder.glob("*.json")) if folder.exists() else []:
        dj = json.loads(p.read_text())
        extra = {k: v for k, v in dj.items() if k not in ("best_params", "model", "target", "best_neg_rmse", "trials", "candidates", "history")
                 and not isinstance(v, (list, dict))}
        rows_c.append(dict(run=run, file=p.stem, **extra, keys=",".join(sorted(dj))))
REPORT["7_tuning_settings_recorded"] = pd.DataFrame(rows_c) if rows_c else "no best_params files found"
cfg_rows = []
for run, d_ in [("LOCO", LOCO_DIR), ("within", WITHIN_DIR)]:
    for f in sorted(d_.glob("*config*.json")):
        try:
            c = json.loads(f.read_text())
            cfg_rows.append(dict(run=run, file=f.name, **{k: c.get(k) for k in ["fast_mode", "n_random_search_iter", "inner_cv_groupkfold_splits",
                                                                                 "dl_n_random_search_iter", "random_seed", "cyclical_period_kyr"]}))
        except Exception as ex:
            cfg_rows.append(dict(run=run, file=f.name, error=str(ex)))
REPORT["7_run_config"] = pd.DataFrame(cfg_rows) if cfg_rows else "no config json found in run folders"
print(REPORT["7_tuning_settings_recorded"]); print(REPORT["7_run_config"])

## 10 · Report — summary of all results

In [ ]:
def _fmt(v):
    if isinstance(v, pd.DataFrame): return v.to_string(index=False) if len(v) else "(empty)"
    if isinstance(v, dict): return "\n".join(f"  {k}: {x}" for k, x in v.items())
    return str(v)
_lines = ["=" * 100, "SISAL REVISION — SUPPLEMENT BUILD REPORT (R3)", f"generated: {pd.Timestamp.now(tz='UTC')}",
          f"output folder: {SUPP}", "=" * 100]
for k, v in REPORT.items():
    if k.startswith("interp_age"): continue
    _lines += ["", f"### {k}", _fmt(v)]
for k in ["S_feature_ablation", "S_gap_duration", "S_abrupt_change", "S_interval_coverage", "S_representativeness"]:
    _lines += ["", f"### table {k}", TABLES[k].to_string(index=False)]
_lines += ["", "### files written", *[f"  {f}" for f in FILES]]
_txt = "\n".join(_lines); (SUPP / "report_nb3.txt").write_text(_txt); print(_txt)
import zipfile
_zip = OUT / f"revision_outputs_{pd.Timestamp.now():%Y%m%d_%H%M}.zip"
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted((SUPP / "figures").glob("*.png")) + sorted(TABS.glob("*.csv")) + \
             [SUPP / "Supplementary_Tables.xlsx", SUPP / "wcbg_heldout_blocks.csv", SUPP / "report_nb3.txt"]:
        if f.exists(): z.write(f, f.relative_to(OUT))
print(f"\n[zip] {_zip}")
if IN_COLAB:
    from google.colab import files; files.download(str(_zip))